# 2. Prompt Management

## Create the Prompt Template in DataRobot (required)

This notebook expects an existing **Prompt Template** in DataRobot. If you see an error like `PromptTemplate ... not found`, it means the `PROMPT_TEMPLATE_ID` in the code cell does not point to a template in *your* tenant.

### Steps (UI)

1. In DataRobot, go to **GenAI / Prompt Templates** (Prompt Management).
2. Click **Create Prompt Template**.
3. Create a **System Prompt** template (recommended) and paste the example below.
4. Save the template.
5. Copy the **Prompt Template ID** from the template URL (or the template details) and paste it into the notebook variable `PROMPT_TEMPLATE_ID`.

### Example system prompt (paste into the template)

```
You are a helpful forecasting assistant.

- Use the forecasting deployment with ID `6971b39b3fa6dde87d114a82`.
- Forecast using the scoring dataset with ID `6971bf6404e148a1b1b17c71`.

```


In [0]:
# --- 1. IMPORTS & SETUP ---
from dotenv import load_dotenv
import datarobot as dr
from datarobot.models.genai.prompt_template import PromptTemplate
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# Load Env & Connect
load_dotenv()
dr_client = dr.Client()
print(f"Connected to DataRobot: {dr_client.endpoint}")

# --- 2. CONFIGURATION ---
# UPDATED: The correct ID from your screenshot URL
PROMPT_TEMPLATE_ID = "6971a9c74f8af03548dc8d24"

# Version to test (Set to "v1", "v2", or None for latest)
PROMPT_VERSION_ID = "v4" 

# Variables matching your screenshot template
RENDER_VARIABLES = {
    "company_name": "DataRobot Forecasting Inc." 
}

# Model Config
MODEL_NAME = "azure/gpt-5-1-2025-11-13" 

# --- 3. FETCH & RENDER ---
print(f"\n--- Fetching Template: {PROMPT_TEMPLATE_ID} ---")
template = PromptTemplate.get(PROMPT_TEMPLATE_ID)
print(f"Template Name: {template.name}")

target_version = None

if PROMPT_VERSION_ID:
    # Logic to handle "v1", "V1", or 1
    versions = template.list_versions()
    search_str = str(PROMPT_VERSION_ID).lower().replace("v", "")
    
    for v in versions:
        # Check exact ID match OR version number match
        if v.id == PROMPT_VERSION_ID:
            target_version = v
            break
        # Safe check for version number
        if hasattr(v, 'version') and str(v.version) == search_str:
            target_version = v
            break
            
    if not target_version:
        available = [f"v{getattr(v, 'version', '?')} (ID: {v.id})" for v in versions]
        raise ValueError(f"Could not find version '{PROMPT_VERSION_ID}'.\nAvailable: {available}")
else:
    print("Fetching latest version...")
    target_version = template.get_latest_version()

print(f"Selected Version: v{getattr(target_version, 'version', '?')} (ID: {target_version.id})")

# Render Prompt
try:
    system_prompt = target_version.render(variables=RENDER_VARIABLES)
    print("\n" + "="*30)
    print("  RENDERED SYSTEM PROMPT")
    print("="*30)
    print(system_prompt)
    print("="*30)
except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    # Fallback to print variables if attribute exists, else generic error
    if hasattr(target_version, 'variables'):
        print(f"REQUIRED Variables: {target_version.variables}")
    print(f"PROVIDED Variables: {list(RENDER_VARIABLES.keys())}")
    raise e

# --- 4. INITIALIZE AGENT ---
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

# Initialize Agent with the rendered prompt
agent = Agent(model=model, system_prompt=system_prompt)

# --- 5. EXECUTION ---
print("\n--- Running Test Query ---")
test_query = "Hello, who are you and who do you work for?"
print(f"User: {test_query}\n")

async with agent:
    response = await agent.run(test_query)
    print(response.output)